# DisorderNet GPU: ESM-2 650M + LoRA on Colab Pro

**Goal:** AUC-ROC > 0.88 on DisProt (5-fold protein-grouped CV)

| Component | Detail |
|---|---|
| Hardware | Colab Pro GPU — A100 (40/80 GB), L4, or T4 (auto-tuned batch size) |
| Backbone | ESM-2 650M (33 layers, 1280-dim) |
| Fine-tuning | LoRA rank 8 on Q/V projections (last 8 layers) |
| Head | 1D CNN + residual skip |
| Data | DisProt (disorder-term filtered annotations) |

### Before you run
1. **Runtime → Change runtime type → GPU** (A100 or L4 recommended)
2. **Runtime → Change runtime type → High-RAM** if available
3. Run all cells top-to-bottom (~12–18 h for full 5-fold CV on A100)

Live progress: tqdm bars per epoch + per-epoch metric lines. Checkpoints saved under `checkpoints/`.

In [ ]:
# ── CELL 1 │ Clone repo (Colab only downloads the notebook by default) ──
import os, sys, subprocess

REPO_URL = "https://github.com/Tommaso-R-Marena/DisorderNet.git"
REPO_DIR = "DisorderNet"

# Already inside the cloned repo (e.g. opened from GitHub file browser)
if os.path.isfile("colab/disordernet_gpu.py"):
    if "." not in sys.path:
        sys.path.insert(0, ".")
    print("✅ Running from repository root.")
elif os.path.isdir(REPO_DIR):
    sys.path.insert(0, REPO_DIR)
    print(f"✅ Repository already at '{REPO_DIR}/'")
else:
    print("Cloning DisorderNet repository...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    sys.path.insert(0, REPO_DIR)
    print("✅ Repository cloned.")

In [ ]:
# ── CELL 2 │ Environment setup ─────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="torch")

from colab.disordernet_gpu import (
    TrainConfig,
    install_dependencies,
    setup_environment,
    fetch_disprot,
    process_disprot,
    print_dataset_summary,
    load_esm_model,
    DisorderNetGPU,
    DisProtDataset,
    disprot_collate,
    run_cross_validation,
    _in_colab,
)

install_dependencies(quiet=True)

# ── Hyperparameters (v2 performance defaults — edit as needed) ─────────
cfg = TrainConfig(
    batch_size=4,          # auto-overridden for your GPU VRAM
    accum_steps=4,
    num_epochs=20,         # was 15
    patience=6,            # was 4
    lora_layers=12,        # was 8 — more ESM adaptation
    lora_rank=16,          # was 8
    lora_on_k=True,        # LoRA on Q/K/V (not just Q/V)
    use_focal_loss=True,   # better on imbalanced + hard residues
    boundary_weight=2.5,   # upweight disorder↔order boundaries
    use_physico_features=True,  # v6-style physics + ESM
    esm_fusion_layers=4,   # fuse last 4 ESM layers (not just layer 33)
    n_folds=5,
    use_gradient_checkpointing=True,
    deterministic=False,   # False = TF32 + cudnn.benchmark (faster)
)

cfg = setup_environment(cfg)
DEVICE = cfg.device
print(f"\nColab: {_in_colab()}  |  Seed: {cfg.seed}")

In [ ]:
# ── CELL 3 │ Optional: mount Google Drive for checkpoint persistence ───
import os

MOUNT_DRIVE = False   # ← set True to save checkpoints across Colab sessions
DRIVE_SUBDIR = "DisorderNet"

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_ckpt = f"/content/drive/MyDrive/{DRIVE_SUBDIR}/checkpoints"
    os.makedirs(drive_ckpt, exist_ok=True)
    cfg.checkpoint_dir = drive_ckpt
    print(f"Checkpoints → {cfg.checkpoint_dir}")
else:
    print("Google Drive not mounted. Checkpoints saved locally in ./checkpoints/")

In [ ]:
# ── CELL 4 │ Download & process DisProt ────────────────────────────────
import json

entries = fetch_disprot(cache_path=cfg.data_cache)
proteins, skipped = process_disprot(entries, cfg)
total_res, total_dis, frac_dis = print_dataset_summary(proteins, skipped)

assert len(proteins) >= 500, f"Expected ≥500 proteins, got {len(proteins)}"
print("\n✅ Data validation passed.")
print("   Disorder labels: disorder-related term_names only.")
print("   Functional regions preserved for biological-utility evaluation.")

In [ ]:
# ── CELL 5 │ Load ESM-2 650M ───────────────────────────────────────────
import torch

model_esm, alphabet, batch_converter = load_esm_model(DEVICE)

mem_gb = torch.cuda.memory_allocated() / 1e9
print(f"  VRAM used: {mem_gb:.1f} GB / {cfg.vram_gb:.1f} GB")

# Quick forward-pass sanity check
_, _, test_tokens = batch_converter([("test", "ACDEFGHIKLMNPQRSTVWY" * 3)])
with torch.inference_mode():
    out = model_esm(test_tokens.to(DEVICE), repr_layers=[33], return_contacts=False)
print(f"  Embedding shape: {out['representations'][33].shape}")
del test_tokens, out
torch.cuda.empty_cache()
print("✅ ESM-2 verified.")

In [ ]:
# ── CELL 6 │ Build model + smoke-test DataLoader ───────────────────────
from torch.utils.data import DataLoader

print("Building DisorderNetGPU...")
model = DisorderNetGPU(model_esm, cfg).to(DEVICE)
print(f"  VRAM after build: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

test_ds = DisProtDataset(proteins[:8], batch_converter)
test_dl = DataLoader(test_ds, batch_size=2, collate_fn=disprot_collate)
tok, lab, msk, ids = next(iter(test_dl))
print(f"  Batch: tokens={tok.shape}  labels={lab.shape}  disorder_frac={lab[msk].mean():.3f}")
del test_ds, test_dl, model
torch.cuda.empty_cache()
print("✅ Model + DataLoader verified.")

In [ ]:
# ── CELL 7 │ 5-Fold Cross-Validation ───────────────────────────────────
# Resume support: if Colab disconnected, set RESUME_FROM_FOLD to the next fold (0-based)
RESUME_FROM_FOLD = 0

from IPython.display import display, HTML, clear_output
import pandas as pd

live_rows = []

def _on_epoch(row):
    live_rows.append(row)
    if len(live_rows) % 1 == 0:
        clear_output(wait=True)
        display(HTML("<b>Latest training metrics</b>"))
        display(pd.DataFrame(live_rows[-10:]))

fold_results, cv_summary = run_cross_validation(
    proteins=proteins,
    esm_backbone=model_esm,
    batch_converter=batch_converter,
    cfg=cfg,
    resume_from_fold=RESUME_FROM_FOLD,
    on_epoch_end=_on_epoch,
)

pooled_auc = cv_summary["pooled_auc"]
pooled_ap  = cv_summary["pooled_ap"]
fold_aucs  = cv_summary["fold_aucs"]
total_cv_h = cv_summary["total_cv_hours"]

all_probs  = __import__("numpy").concatenate([r["val_probs"] for r in fold_results])
all_labels = __import__("numpy").concatenate([r["val_labels"] for r in fold_results])

with open("cv_results.json", "w") as f:
    json.dump(cv_summary, f, indent=2)
print("\nSaved cv_results.json")

if pooled_auc >= 0.88:
    print(f"\n🎉 TARGET AUC ≥ 0.88 ACHIEVED! ({pooled_auc:.4f})")
elif pooled_auc >= 0.83:
    print(f"\n✅ Beats DisorderNet v6 baseline (0.831). AUC = {pooled_auc:.4f}")

In [ ]:
# ── CELL 8 │ CAID metrics + matched benchmark tables ───────────────────
from colab.colab_figures import optimal_threshold
from colab.benchmark_tables import print_matched_benchmark_report
from colab.caid_reporting import run_full_caid_report, print_caid_report, save_caid_report
from sklearn.metrics import f1_score, matthews_corrcoef

our_auc = pooled_auc
our_ap  = pooled_ap
opt_thresh, _ = optimal_threshold(all_labels, all_probs)
preds_opt = (all_probs >= opt_thresh).astype(int)
our_f1  = f1_score(all_labels.astype(int), preds_opt)
our_mcc = matthews_corrcoef(all_labels.astype(int), preds_opt)

caid_report = run_full_caid_report(
    proteins=proteins,
    fold_results=fold_results,
    threshold=opt_thresh,
    n_folds=cfg.n_folds,
)
print_caid_report(caid_report)
save_caid_report(caid_report, "caid_evaluation_report.json")

f1_max = caid_report["stratified"]["pooled"].get("f1_max")
benchmark_report = print_matched_benchmark_report(
    gpu_auc=our_auc,
    gpu_ap=our_ap,
    gpu_f1_max=f1_max,
    gpu_mcc=our_mcc,
)

print(f"\n  F1@opt={our_f1:.4f}  MCC@opt={our_mcc:.4f}  F1_max={f1_max:.4f}  opt_thresh={opt_thresh:.3f}")
print("  ✅ Saved caid_evaluation_report.json")

In [ ]:
# ── CELL 9 │ Biological utility report (Phase 1) ─────────────────────────
from colab.biological_utility import (
    run_biological_utility_report,
    print_biological_utility_report,
    save_biological_utility_report,
)
from colab.colab_figures import generate_biological_utility_figure

bio_report = run_biological_utility_report(
    proteins=proteins,
    fold_results=fold_results,
    threshold=opt_thresh,
    n_folds=cfg.n_folds,
)
print_biological_utility_report(bio_report)
save_biological_utility_report(bio_report, "biological_utility_report.json")
generate_biological_utility_figure(bio_report)
print("\n✅ Saved biological_utility_report.json and fig5_biological_utility.{pdf,png}")

In [ ]:
# ── CELL 10 │ Phase 2: AlphaFold pLDDT hallucination rescue ────────────
from colab.af_hallucination import (
    fetch_and_run_af_rescue_report,
    print_af_rescue_report,
    save_af_rescue_report,
)
from colab.colab_figures import generate_af_rescue_figure

HIGH_PLDDT = 70.0   # AF "confident" threshold; disordered + high pLDDT = hallucination
AF_CACHE_DIR = "af_plddt_cache"

af_report, plddt_by_protein = fetch_and_run_af_rescue_report(
    proteins=proteins,
    fold_results=fold_results,
    threshold=opt_thresh,
    high_plddt_threshold=HIGH_PLDDT,
    n_folds=cfg.n_folds,
    cache_dir=AF_CACHE_DIR,
    sleep_s=0.05,
)
print_af_rescue_report(af_report)
save_af_rescue_report(af_report, "af_rescue_report.json")
generate_af_rescue_figure(af_report)
print("\n✅ Saved af_rescue_report.json and fig6_af_rescue.{pdf,png}")

In [ ]:
# ── CELL 11 │ Phase 2b: AlphaFold 3 via Google Drive (optional) ─────────
# AF3 weights CANNOT go on GitHub (license + size). Use Drive:
#   MyDrive/DisorderNet/af3/af3.bin
#   MyDrive/DisorderNet/af3/outputs/<protein_job_folders>/
#
# Modes:
#   "off"    — skip AF3 (AF2-only from Cell 10)
#   "ingest" — load precomputed AF3 outputs from Drive (recommended)
#   "run"    — run AF3 on a small subset on Colab A100 (needs alphafold3 clone)

from colab.af3_colab import (
    DEFAULT_DRIVE_ROOT,
    print_af3_setup_instructions,
    run_af3_subset_on_colab,
    setup_af3_for_colab,
)
from colab.af_hallucination import (
    fetch_and_run_af3_rescue_report,
    run_af2_af3_comparison_report,
    print_af2_af3_comparison,
    print_af_rescue_report,
    save_af2_af3_comparison,
    save_af_rescue_report,
)
from colab.colab_figures import generate_af2_af3_comparison_figure

AF3_MODE = "ingest"  # "off" | "ingest" | "run"
AF3_DRIVE_ROOT = DEFAULT_DRIVE_ROOT
AF3_MAX_RUN = 25     # only if AF3_MODE == "run"
AF3_CACHE_DIR = "af3_plddt_cache"

af3_report = {"insufficient_data": True, "skipped": True}
af3_comparison = {"insufficient_data": True}
plddt_af3_by_protein = {}

af3_cfg = setup_af3_for_colab(mode=AF3_MODE, drive_root=AF3_DRIVE_ROOT, mount_drive=True)
print_af3_setup_instructions(af3_cfg["paths"])

if AF3_MODE == "off":
    print("AF3 disabled (AF3_MODE='off'). Using AF2 results from Cell 10 only.")
elif AF3_MODE == "run":
    if not af3_cfg.get("ready"):
        print("⚠️ AF3 run prerequisites missing:", af3_cfg.get("weights_message", ""))
    else:
        run_result = run_af3_subset_on_colab(
            proteins=proteins,
            paths=af3_cfg["paths"],
            max_proteins=AF3_MAX_RUN,
            run_data_pipeline=False,
        )
        print("AF3 subset run:", run_result)
        if not run_result.get("success"):
            print("⚠️ AF3 run failed — switch to ingest mode with pre-uploaded outputs.")

if AF3_MODE in ("ingest", "run") and af3_cfg["paths"]["output_dir"]:
    af3_report, plddt_af3_by_protein = fetch_and_run_af3_rescue_report(
        proteins=proteins,
        fold_results=fold_results,
        af3_output_root=af3_cfg["paths"]["output_dir"],
        threshold=opt_thresh,
        high_plddt_threshold=HIGH_PLDDT,
        n_folds=cfg.n_folds,
        cache_dir=AF3_CACHE_DIR,
    )
    af3_report["skipped"] = False
    print_af_rescue_report(af3_report)
    save_af_rescue_report(af3_report, "af3_rescue_report.json")

    af3_comparison = run_af2_af3_comparison_report(af_report, af3_report)
    print_af2_af3_comparison(af3_comparison)
    save_af2_af3_comparison(af3_comparison, "af2_af3_comparison.json")
    generate_af2_af3_comparison_figure(af3_comparison, af2_report=af_report, af3_report=af3_report)
    print("\n✅ Saved af3_rescue_report.json, af2_af3_comparison.json, fig7_af2_af3_comparison.{pdf,png}")
else:
    print("No AF3 outputs configured — upload predictions to Drive or set AF3_MODE='run'.")

In [ ]:
# ── CELL 12 │ Phase 3: Structure calibration & integrated synthesis ────
from colab.phase3_synthesis import (
    run_structure_calibration_report,
    run_phase3_integrated_report,
    print_phase3_report,
    save_phase3_report,
)
from colab.statistical_validation import (
    run_full_statistical_validation,
    print_statistical_validation,
    save_statistical_validation,
)
from colab.colab_figures import generate_phase3_figure

calibration_report = run_structure_calibration_report(
    proteins=proteins,
    fold_results=fold_results,
    plddt_by_protein=plddt_by_protein,
    threshold=opt_thresh,
    high_plddt_threshold=HIGH_PLDDT,
    n_folds=cfg.n_folds,
    n_boot=500,
    af_source="AlphaFold DB (AF2)",
)

stat_validation = run_full_statistical_validation(
    proteins=proteins,
    fold_results=fold_results,
    plddt_by_protein=plddt_by_protein,
    n_folds=cfg.n_folds,
)
print_statistical_validation(stat_validation)
save_statistical_validation(stat_validation, "statistical_validation_report.json")

cv_pooled = {
    "auc": float(our_auc),
    "ap": float(our_ap),
    "f1": float(our_f1),
    "mcc": float(our_mcc),
    "opt_threshold": float(opt_thresh),
    "f1_max": caid_report["stratified"]["pooled"].get("f1_max"),
}

af3_for_phase3 = af3_report if not af3_report.get("skipped") else None
comparison_for_phase3 = (
    af3_comparison if not af3_comparison.get("insufficient_data") else None
)

phase3_report = run_phase3_integrated_report(
    cv_pooled=cv_pooled,
    bio_report=bio_report,
    af_report=af_report,
    calibration_report=calibration_report,
    af3_report=af3_for_phase3,
    af2_af3_comparison=comparison_for_phase3,
    caid_report=caid_report,
    statistical_validation=stat_validation,
    benchmark_report=benchmark_report,
)
print_phase3_report(phase3_report)
save_phase3_report(phase3_report, "phase3_integrated_report.json")
generate_phase3_figure(phase3_report)
print("\n✅ Saved phase3_integrated_report.json, statistical_validation_report.json, fig8_phase3_synthesis.{pdf,png}")

In [ ]:
# ── CELL 13 │ Figures ──────────────────────────────────────────────────
from colab.colab_figures import generate_all_figures

fig_metrics = generate_all_figures(
    fold_results, all_labels, all_probs, our_auc, our_ap
)
opt_thresh = fig_metrics["opt_thresh"]
our_f1     = fig_metrics["f1"]
our_mcc    = fig_metrics["mcc"]
print("✅ All figures saved.")

In [ ]:
# ── CELL 14 │ Save results & best checkpoint ───────────────────────────
import shutil
import numpy as np
from datetime import datetime

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
best_fold_idx = int(np.argmax([r["best_auc"] for r in fold_results]))
best_fold     = fold_results[best_fold_idx]
best_ckpt_dst = f"disordernet_gpu_best_{RUN_TIMESTAMP}.pt"

if best_fold.get("ckpt_path") and os.path.exists(best_fold["ckpt_path"]):
    shutil.copy(best_fold["ckpt_path"], best_ckpt_dst)
    print(f"Best checkpoint: {best_ckpt_dst}  (fold {best_fold['fold']}, AUC={best_fold['best_auc']:.4f})")

full_results = {
    "run_timestamp": RUN_TIMESTAMP,
    "config": cv_summary["config"],
    "dataset": {"n_proteins": len(proteins), "n_residues": int(total_res), "frac_disorder": float(frac_dis)},
    "pooled": {"auc": float(our_auc), "ap": float(our_ap), "f1": float(our_f1),
               "mcc": float(our_mcc), "opt_threshold": float(opt_thresh)},
    "per_fold": [{"fold": r["fold"], "best_auc": r["best_auc"], "best_ap": r["best_ap"],
                  "train_time_h": r["total_time"] / 3600, "history": r["history"]} for r in fold_results],
    "best_checkpoint": best_ckpt_dst,
    "biological_utility": bio_report,
    "af_rescue": af_report,
    "af3_rescue": af3_report,
    "af2_af3_comparison": af3_comparison,
    "caid_evaluation": caid_report,
    "statistical_validation": stat_validation,
    "structure_calibration": calibration_report,
    "phase3_integrated": phase3_report,
    "benchmark_tables": benchmark_report,
}
results_path = f"disordernet_gpu_results_{RUN_TIMESTAMP}.json"
with open(results_path, "w") as f:
    json.dump(full_results, f, indent=2)

print(f"\n{'═'*60}")
print(" DISORDERNET GPU — FINAL RESULTS")
print(f"{'═'*60}")
print(f"  AUC-ROC  : {our_auc:.4f}")
print(f"  Avg Prec : {our_ap:.4f}")
print(f"  F1       : {our_f1:.4f}")
print(f"  Total CV : {total_cv_h:.2f} h")
print(f"  Saved    : {results_path}")
print(f"{'═'*60}")